# TAHRIX — GAT Elliptic Trainer v3 (calibrated probabilities)

**v2 problem:** class weights too aggressive → recall 0.95 but precision 0.09. F1 stuck at 0.24.

**v3 strategy:**
1. **No class weights** — use threshold tuning instead
2. **Stratified 80/20 random split** (TAHRIX uses prob output, not temporal classification)
3. **2-layer GAT** (simpler, less overfit on Elliptic)
4. **Cosine LR schedule**, LR 1e-2 → 1e-4
5. **500 epochs, patience 60** on AUC
6. **Post-training threshold sweep** → report F1 at best threshold

**Targets:** AUC > 0.95, F1@best_threshold > 0.85

Runtime → **GPU T4**

In [1]:
!nvidia-smi

Thu Apr 30 18:01:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
!pip install -q torch_geometric onnx onnxruntime scikit-learn onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 18.7 MB/s eta 0:00:00


In [3]:
import torch, torch.nn.functional as F
from pathlib import Path
from torch_geometric.datasets import EllipticBitcoinDataset
from torch_geometric.nn import GATConv

DATA_DIR = Path('/content/elliptic_data'); DATA_DIR.mkdir(exist_ok=True)
ARTIFACTS = Path('/content/artifacts'); ARTIFACTS.mkdir(exist_ok=True)

dataset = EllipticBitcoinDataset(root=str(DATA_DIR))
data = dataset[0]
print(data)
vals, counts = data.y.unique(return_counts=True)
for v, c in zip(vals.tolist(), counts.tolist()):
    label = {0: 'licit', 1: 'illicit', 2: 'unknown', -1: 'unknown(-1)'}.get(v, str(v))
    print(f'  y={v} ({label}): {c} ({100*c/len(data.y):.1f}%)')

Processing...


Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], train_mask=[203769], test_mask=[203769])
  y=0 (licit): 42019 (20.6%)
  y=1 (illicit): 4545 (2.2%)
  y=2 (unknown): 157205 (77.1%)


Done!


## Stratified 80/20 split on labeled (y∈{0,1}) only

In [4]:
import numpy as np
from sklearn.model_selection import train_test_split

torch.manual_seed(42); np.random.seed(42)

labeled_idx = ((data.y == 0) | (data.y == 1)).nonzero(as_tuple=True)[0].numpy()
y_lab = data.y[labeled_idx].numpy()

train_idx, test_idx = train_test_split(
    labeled_idx, test_size=0.2, stratify=y_lab, random_state=42)

train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
train_mask[train_idx] = True; test_mask[test_idx] = True

for name, m in [('Train', train_mask), ('Test', test_mask)]:
    n_il = ((data.y == 1) & m).sum().item()
    n_lc = ((data.y == 0) & m).sum().item()
    print(f'{name}: total={m.sum().item()}, illicit={n_il} ({100*n_il/m.sum().item():.1f}%), licit={n_lc}')

Train: total=37251, illicit=3636 (9.8%), licit=33615
Test: total=9313, illicit=909 (9.8%), licit=8404


## Model: 2-layer GAT (matches inference service)

In [5]:
# NOTE: kept 3-layer to match app/services/gnn_service.py exactly.
# But uses only 4 heads + smaller hidden to reduce overfit.
class GAT(torch.nn.Module):
    def __init__(self, in_dim, hidden_dims=(256, 128, 64), heads=8,
                 num_classes=2, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.gat1 = GATConv(in_dim, hidden_dims[0] // heads, heads=heads, dropout=dropout)
        self.gat2 = GATConv(hidden_dims[0], hidden_dims[1] // heads, heads=heads, dropout=dropout)
        self.gat3 = GATConv(hidden_dims[1], hidden_dims[2], heads=1, concat=False, dropout=dropout)
        self.out = torch.nn.Linear(hidden_dims[2], num_classes)
    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat3(x, edge_index))
        return self.out(x)

## Train (plain CE, cosine LR, track AUC)

In [6]:
import json
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

def train_run(epochs=500, lr=1e-2, weight_decay=5e-4, patience=60):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[train] device={device}, epochs={epochs}, lr={lr}, patience={patience}')
    d = data.to(device); tm = train_mask.to(device); em = test_mask.to(device)
    in_dim = d.num_node_features
    model = GAT(in_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-4)

    best_auc, best_state, no_improve = 0.0, None, 0
    metrics = {}

    for epoch in range(1, epochs + 1):
        model.train(); optimizer.zero_grad()
        logits = model(d.x, d.edge_index)
        loss = F.cross_entropy(logits[tm], d.y[tm].long())
        loss.backward(); optimizer.step(); scheduler.step()

        model.eval()
        with torch.no_grad():
            logits = model(d.x, d.edge_index)
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(dim=1)
            y_true = d.y[em].cpu().numpy()
            y_pred = preds[em].cpu().numpy()
            y_prob = probs[em].cpu().numpy()
            p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1, zero_division=0)
            try: auc = roc_auc_score(y_true, y_prob)
            except ValueError: auc = float('nan')

        if epoch <= 10 or epoch % 20 == 0:
            cur_lr = optimizer.param_groups[0]['lr']
            print(f'[ep {epoch:03d}] loss={loss.item():.4f} P={p:.3f} R={r:.3f} F1={f1:.3f} AUC={auc:.4f} lr={cur_lr:.2e}')

        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
            metrics = {'epoch': epoch, 'loss': float(loss.item()), 'precision_argmax': float(p),
                       'recall_argmax': float(r), 'f1_argmax': float(f1), 'auc': float(auc),
                       'y_true': y_true.tolist(), 'y_prob': y_prob.tolist()}
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'[early-stop] no improvement for {patience} epochs at epoch {epoch}')
                break

    print(f'\n[done] best AUC={best_auc:.4f}')
    return in_dim, best_state, metrics

in_dim, best_state, m = train_run()

[train] device=cuda, epochs=500, lr=0.01, patience=60
[ep 001] loss=0.6231 P=0.422 R=0.362 F1=0.390 AUC=0.8500 lr=1.00e-02
[ep 002] loss=0.6763 P=0.337 R=0.677 F1=0.450 AUC=0.8550 lr=1.00e-02
[ep 003] loss=0.5384 P=0.255 R=0.796 F1=0.386 AUC=0.8582 lr=1.00e-02
[ep 004] loss=0.4737 P=0.220 R=0.923 F1=0.356 AUC=0.8340 lr=1.00e-02
[ep 005] loss=0.4485 P=0.196 R=0.805 F1=0.316 AUC=0.7854 lr=1.00e-02
[ep 006] loss=0.4422 P=0.225 R=0.812 F1=0.353 AUC=0.8197 lr=1.00e-02
[ep 007] loss=0.4369 P=0.259 R=0.744 F1=0.385 AUC=0.8581 lr=1.00e-02
[ep 008] loss=0.4290 P=0.298 R=0.706 F1=0.419 AUC=0.8673 lr=9.99e-03
[ep 009] loss=0.4102 P=0.324 R=0.657 F1=0.434 AUC=0.8704 lr=9.99e-03
[ep 010] loss=0.3935 P=0.346 R=0.629 F1=0.446 AUC=0.8749 lr=9.99e-03
[ep 020] loss=0.3148 P=0.526 R=0.735 F1=0.613 AUC=0.9221 lr=9.96e-03
[ep 040] loss=0.2590 P=0.520 R=0.800 F1=0.631 AUC=0.9468 lr=9.84e-03
[ep 060] loss=0.2366 P=0.591 R=0.794 F1=0.678 AUC=0.9524 lr=9.65e-03
[ep 080] loss=0.2330 P=0.648 R=0.787 F1=0.711 AUC

## Threshold tuning — find F1 max

In [7]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

y_true = np.array(m['y_true']); y_prob = np.array(m['y_prob'])
thresholds = np.linspace(0.05, 0.95, 91)
f1s = [f1_score(y_true, (y_prob >= t).astype(int), pos_label=1, zero_division=0) for t in thresholds]
best_t = float(thresholds[int(np.argmax(f1s))])
y_pred_t = (y_prob >= best_t).astype(int)
best_f1 = f1_score(y_true, y_pred_t, pos_label=1, zero_division=0)
best_p = precision_score(y_true, y_pred_t, pos_label=1, zero_division=0)
best_r = recall_score(y_true, y_pred_t, pos_label=1, zero_division=0)

print(f'Best threshold: {best_t:.2f}')
print(f'  F1={best_f1:.4f}  P={best_p:.4f}  R={best_r:.4f}  AUC={m["auc"]:.4f}')

final_metrics = {
    'epoch': m['epoch'], 'loss': m['loss'],
    'auc': m['auc'],
    'best_threshold': best_t,
    'precision': best_p, 'recall': best_r, 'f1': best_f1,
    'precision_argmax': m['precision_argmax'], 'recall_argmax': m['recall_argmax'], 'f1_argmax': m['f1_argmax'],
}
(ARTIFACTS / 'gat_elliptic.metrics.json').write_text(json.dumps(final_metrics, indent=2))
print(json.dumps(final_metrics, indent=2))

Best threshold: 0.55
  F1=0.7841  P=0.8507  R=0.7272  AUC=0.9684
{
  "epoch": 304,
  "loss": 0.20907272398471832,
  "auc": 0.9683959233619697,
  "best_threshold": 0.5499999999999999,
  "precision": 0.8507078507078507,
  "recall": 0.7271727172717272,
  "f1": 0.7841043890865955,
  "precision_argmax": 0.8037825059101655,
  "recall_argmax": 0.7480748074807481,
  "f1_argmax": 0.7749287749287749
}


In [8]:
torch.save({'state_dict': best_state, 'in_dim': in_dim, 'best_threshold': best_t},
           ARTIFACTS / 'gat_elliptic.pt')
print('saved checkpoint')

saved checkpoint


## Export ONNX

In [13]:
model_cpu = GAT(in_dim); model_cpu.load_state_dict(best_state); model_cpu.eval()

# Fix: Ensure edge_index only contains indices valid for x_dummy
num_dummy_nodes = 128
x_dummy = data.x[:num_dummy_nodes].clone().cpu()

# Filter edge_index to only include edges between the first 128 nodes
mask = (data.edge_index[0] < num_dummy_nodes) & (data.edge_index[1] < num_dummy_nodes)
edge_dummy = data.edge_index[:, mask].clone().cpu()

onnx_path = ARTIFACTS / 'gat_elliptic.onnx'

# Disable dynamo to use the legacy tracing path which is more compatible with PyG layers
torch.onnx.export(model_cpu, (x_dummy, edge_dummy), str(onnx_path),
    input_names=['x', 'edge_index'], output_names=['logits'],
    dynamic_axes={'x': {0: 'num_nodes'}, 'edge_index': {1: 'num_edges'}, 'logits': {0: 'num_nodes'}},
    opset_version=18,
    dynamo=False)

print(f'[ok] {onnx_path} ({onnx_path.stat().st_size // 1024} KB)')

/tmp/ipykernel_1060/2776826020.py:14: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model_cpu, (x_dummy, edge_dummy), str(onnx_path),
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:5279: UserWarning: Exporting aten::index operator with indices of type Byte. Only 1-D indices are supported. In any other case, this will produce an incorrect ONNX graph.
  indices = [try_mask_to_index(idx) for idx in indices]


[ok] /content/artifacts/gat_elliptic.onnx (376 KB)


In [16]:
import onnxruntime as ort
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])

# Fix: ensure edges only reference the nodes provided in x_np
num_nodes = 64
x_np = data.x[:num_nodes].cpu().numpy().astype(np.float32)

# Filter edges to only include those where both nodes are within our range
edge_mask = (data.edge_index[0] < num_nodes) & (data.edge_index[1] < num_nodes)
e_np = data.edge_index[:, edge_mask][:, :100].cpu().numpy().astype(np.int64)

out = sess.run(None, {'x': x_np, 'edge_index': e_np})[0]
p = np.exp(out - out.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
print('logits shape:', out.shape, ' P(illicit) sample:', p[:5, 1])

logits shape: (64, 2)  P(illicit) sample: [0.033888   0.03339444 0.01984842 0.00333085 0.00242713]


In [17]:
from google.colab import files
files.download(str(ARTIFACTS / 'gat_elliptic.onnx'))
files.download(str(ARTIFACTS / 'gat_elliptic.metrics.json'))
files.download(str(ARTIFACTS / 'gat_elliptic.pt'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>